#Car_Sales_Advanced_Starter

### SETUP + DATA LOADING

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

#### Base Path

In [0]:
base_path = "/Volumes/workspace/default/day6_files/"

#### Load CSV Files

In [0]:
customers = spark.read.option("header", True).csv(base_path + "customers.csv")
cars = spark.read.option("header", True).csv(base_path + "cars.csv")
sales = spark.read.option("header", True).csv(base_path + "sales.csv")
dealers = spark.read.option("header", True).csv(base_path + "dealers.csv")
sales_dealer = spark.read.option("header", True).csv(base_path + "sales_dealer.csv")

#### Fix Data Types

In [0]:
cars = cars.withColumn("price", col("price").cast("int"))

sales = sales \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("sale_date", to_date(col("sale_date")))

### PHASE 1 – DATA UNDERSTANDING

In [0]:
print("=" * 60)
print("PHASE 1 – DATA UNDERSTANDING")
print("=" * 60)

PHASE 1 – DATA UNDERSTANDING

--- Schema & Row Counts ---
customers: 50 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

cars: 30 rows
root
 |-- car_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: integer (nullable = true)

sales: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- car_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)

dealers: 10 rows
root
 |-- dealer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

sales_dealer: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- dealer_id: string (nullable = true)


--- Null Values per Column ---

customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   0|   0|
+-----------+----+----+


c

#### Schema & Row Counts

In [0]:
print("\n--- Schema & Row Counts ---")
for label, df in [("customers", customers), ("cars", cars),
                  ("sales", sales), ("dealers", dealers), ("sales_dealer", sales_dealer)]:
    print(f"{label}: {df.count()} rows")
    df.printSchema()

PHASE 1 – DATA UNDERSTANDING

--- Schema & Row Counts ---
customers: 50 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

cars: 30 rows
root
 |-- car_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: integer (nullable = true)

sales: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- car_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)

dealers: 10 rows
root
 |-- dealer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

sales_dealer: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- dealer_id: string (nullable = true)


--- Null Values per Column ---

customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   0|   0|
+-----------+----+----+


c

#### Null Values Check

In [0]:
print("\n--- Null Values per Column ---")
for label, df in [("customers", customers), ("cars", cars), ("sales", sales),
                  ("dealers", dealers), ("sales_dealer", sales_dealer)]:
    null_counts = df.select([
        sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
    ])
    print(f"\n{label}:")
    null_counts.show()

PHASE 1 – DATA UNDERSTANDING

--- Schema & Row Counts ---
customers: 50 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

cars: 30 rows
root
 |-- car_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: integer (nullable = true)

sales: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- car_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)

dealers: 10 rows
root
 |-- dealer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

sales_dealer: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- dealer_id: string (nullable = true)


--- Null Values per Column ---

customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   0|   0|
+-----------+----+----+


c

#### Duplicate Rows

In [0]:
print("\n--- Duplicate Rows ---")
for label, df in [("customers", customers), ("cars", cars), ("sales", sales)]:
    total = df.count()
    distinct = df.distinct().count()
    print(f"{label}: {total - distinct} duplicate rows")

PHASE 1 – DATA UNDERSTANDING

--- Schema & Row Counts ---
customers: 50 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

cars: 30 rows
root
 |-- car_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: integer (nullable = true)

sales: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- car_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)

dealers: 10 rows
root
 |-- dealer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

sales_dealer: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- dealer_id: string (nullable = true)


--- Null Values per Column ---

customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   0|   0|
+-----------+----+----+


c

#### Invalid Values: Negative Prices

In [0]:
print("\n--- Cars with Negative Prices ---")
cars.filter(col("price") < 0).show()

PHASE 1 – DATA UNDERSTANDING

--- Schema & Row Counts ---
customers: 50 rows
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

cars: 30 rows
root
 |-- car_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: integer (nullable = true)

sales: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- car_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)

dealers: 10 rows
root
 |-- dealer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

sales_dealer: 80 rows
root
 |-- sale_id: string (nullable = true)
 |-- dealer_id: string (nullable = true)


--- Null Values per Column ---

customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   0|   0|
+-----------+----+----+


c

### PHASE 2 – CLEANING

In [0]:
print("=" * 60)
print("PHASE 2 – CLEANING")
print("=" * 60)

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### Trim String Columns

In [0]:
customers_clean = customers \
    .withColumn("name", trim(col("name"))) \
    .withColumn("city", trim(col("city")))

dealers_clean = dealers \
    .withColumn("name", trim(col("name"))) \
    .withColumn("city", trim(col("city")))

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### Fix Negative Prices → Absolute Value

In [0]:
cars_clean = cars.withColumn("price", abs(col("price")))

print("\n--- Cars after fixing negative prices ---")
cars_clean.filter(col("car_id").isin(102, 104, 107, 108, 109, 116, 119)).show()

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### Parse sale_date

In [0]:
sales_clean = sales.withColumn("sale_date", to_date(col("sale_date")))

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### Invalid Foreign Keys Detection

In [0]:
invalid_cust_ids = sales_clean \
    .join(customers_clean, "customer_id", "left_anti") \
    .select("customer_id").distinct()

print("\n--- Invalid customer_ids in sales ---")
invalid_cust_ids.show()

invalid_car_ids = sales_clean \
    .join(cars_clean, "car_id", "left_anti") \
    .select("car_id").distinct()

print("\n--- Invalid car_ids in sales ---")
invalid_car_ids.show()

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### Remove Invalid Foreign Keys

In [0]:
sales_clean = sales_clean \
    .join(customers_clean.select("customer_id"), "customer_id", "inner") \
    .join(cars_clean.select("car_id"), "car_id", "inner")

print(f"\nSales rows after cleaning: {sales_clean.count()}")

PHASE 2 – CLEANING

--- Cars after fixing negative prices ---
+------+-------+--------+-----+
|car_id|  brand|   model|price|
+------+-------+--------+-----+
|   102|  Honda|Model102| 6970|
|   104|  Honda|Model104|16650|
|   107|Hyundai|Model107| 1271|
|   108| Toyota|Model108|18630|
|   109|Hyundai|Model109|  613|
|   116| Toyota|Model116|  488|
|   119| Toyota|Model119| 8231|
+------+-------+--------+-----+


--- Invalid customer_ids in sales ---
+-----------+
|customer_id|
+-----------+
|         56|
|         55|
|         52|
|         60|
|         59|
|         51|
|         57|
|         58|
|         54|
|         53|
+-----------+


--- Invalid car_ids in sales ---
+------+
|car_id|
+------+
+------+


Sales rows after cleaning: 63


#### PHASE 3 – VALIDATION REPORT

In [0]:
print("=" * 60)
print("PHASE 3 – VALIDATION REPORT")
print("=" * 60)

PHASE 3 – VALIDATION REPORT

--- Validation Summary ---
Invalid customer_id : 17
Invalid car_id      : 0
Invalid sale_id     : 17


#### Orphan Records

In [0]:
orphan_sales_cust = sales.join(customers_clean, "customer_id", "left_anti")
orphan_sales_cars = sales.join(cars_clean, "car_id", "left_anti")
orphan_sd = sales_dealer.join(sales_clean, "sale_id", "left_anti")

print("\n--- Validation Summary ---")
print(f"Invalid customer_id : {orphan_sales_cust.count()}")
print(f"Invalid car_id      : {orphan_sales_cars.count()}")
print(f"Invalid sale_id     : {orphan_sd.count()}")

PHASE 3 – VALIDATION REPORT

--- Validation Summary ---
Invalid customer_id : 17
Invalid car_id      : 0
Invalid sale_id     : 17


### PHASE 4 – TRANSFORMATIONS

In [0]:
print("=" * 60)
print("PHASE 4 – TRANSFORMATIONS")
print("=" * 60)

PHASE 4 – TRANSFORMATIONS

--- Customer Revenue ---
+-----------+------+---------+-------------+-----------+-----------+
|customer_id|  name|     city|total_revenue|total_sales|total_units|
+-----------+------+---------+-------------+-----------+-----------+
|         24|Cust24|Hyderabad|       233449|          3|          7|
|         13|Cust13|  Chennai|       230502|          5|         11|
|         17|Cust17|  Chennai|       220322|          3|          7|
|         34|Cust34|  Chennai|       213216|          2|          5|
|         41|Cust41|   Mumbai|       154246|          2|          4|
|         23|Cust23|Hyderabad|       147509|          3|          6|
|         50|Cust50|Hyderabad|       144497|          2|          5|
|         37|Cust37|Bangalore|       135511|          2|          4|
|          4| Cust4|   Mumbai|       133266|          2|          4|
|          5| Cust5|Bangalore|       126977|          2|          3|
+-----------+------+---------+-------------+-------

#### Build Final Dataset

In [0]:
df = sales_clean \
    .join(customers_clean, "customer_id") \
    .join(cars_clean, "car_id") \
    .join(sales_dealer, "sale_id") \
    .join(dealers_clean.withColumnRenamed("name", "dealer_name")
                       .withColumnRenamed("city", "dealer_city"), "dealer_id") \
    .withColumn("revenue", col("price") * col("quantity"))

PHASE 4 – TRANSFORMATIONS

--- Customer Revenue ---
+-----------+------+---------+-------------+-----------+-----------+
|customer_id|  name|     city|total_revenue|total_sales|total_units|
+-----------+------+---------+-------------+-----------+-----------+
|         24|Cust24|Hyderabad|       233449|          3|          7|
|         13|Cust13|  Chennai|       230502|          5|         11|
|         17|Cust17|  Chennai|       220322|          3|          7|
|         34|Cust34|  Chennai|       213216|          2|          5|
|         41|Cust41|   Mumbai|       154246|          2|          4|
|         23|Cust23|Hyderabad|       147509|          3|          6|
|         50|Cust50|Hyderabad|       144497|          2|          5|
|         37|Cust37|Bangalore|       135511|          2|          4|
|          4| Cust4|   Mumbai|       133266|          2|          4|
|          5| Cust5|Bangalore|       126977|          2|          3|
+-----------+------+---------+-------------+-------

#### 4.1 Customer Revenue

In [0]:
print("\n--- Customer Revenue ---")
customer_revenue = df.groupBy("customer_id", "name", "city") \
    .agg(sum("revenue").alias("total_revenue"),
         count("sale_id").alias("total_sales"),
         sum("quantity").alias("total_units")) \
    .orderBy(col("total_revenue").desc())

customer_revenue.show(10)

PHASE 4 – TRANSFORMATIONS

--- Customer Revenue ---
+-----------+------+---------+-------------+-----------+-----------+
|customer_id|  name|     city|total_revenue|total_sales|total_units|
+-----------+------+---------+-------------+-----------+-----------+
|         24|Cust24|Hyderabad|       233449|          3|          7|
|         13|Cust13|  Chennai|       230502|          5|         11|
|         17|Cust17|  Chennai|       220322|          3|          7|
|         34|Cust34|  Chennai|       213216|          2|          5|
|         41|Cust41|   Mumbai|       154246|          2|          4|
|         23|Cust23|Hyderabad|       147509|          3|          6|
|         50|Cust50|Hyderabad|       144497|          2|          5|
|         37|Cust37|Bangalore|       135511|          2|          4|
|          4| Cust4|   Mumbai|       133266|          2|          4|
|          5| Cust5|Bangalore|       126977|          2|          3|
+-----------+------+---------+-------------+-------

#### 4.2 Brand-wise Sales

In [0]:
print("\n--- Brand-wise Sales ---")
brand_sales = df.groupBy("brand") \
    .agg(sum("revenue").alias("total_revenue"),
         count("sale_id").alias("total_sales"),
         sum("quantity").alias("total_units"),
         avg("price").alias("avg_price"))

brand_sales.show()

PHASE 4 – TRANSFORMATIONS

--- Customer Revenue ---
+-----------+------+---------+-------------+-----------+-----------+
|customer_id|  name|     city|total_revenue|total_sales|total_units|
+-----------+------+---------+-------------+-----------+-----------+
|         24|Cust24|Hyderabad|       233449|          3|          7|
|         13|Cust13|  Chennai|       230502|          5|         11|
|         17|Cust17|  Chennai|       220322|          3|          7|
|         34|Cust34|  Chennai|       213216|          2|          5|
|         41|Cust41|   Mumbai|       154246|          2|          4|
|         23|Cust23|Hyderabad|       147509|          3|          6|
|         50|Cust50|Hyderabad|       144497|          2|          5|
|         37|Cust37|Bangalore|       135511|          2|          4|
|          4| Cust4|   Mumbai|       133266|          2|          4|
|          5| Cust5|Bangalore|       126977|          2|          3|
+-----------+------+---------+-------------+-------

#### 4.3 City-wise Revenue

In [0]:
print("\n--- City-wise Revenue ---")
city_revenue = df.groupBy("city") \
    .agg(sum("revenue").alias("total_revenue"))

city_revenue.show()

PHASE 4 – TRANSFORMATIONS

--- Customer Revenue ---
+-----------+------+---------+-------------+-----------+-----------+
|customer_id|  name|     city|total_revenue|total_sales|total_units|
+-----------+------+---------+-------------+-----------+-----------+
|         24|Cust24|Hyderabad|       233449|          3|          7|
|         13|Cust13|  Chennai|       230502|          5|         11|
|         17|Cust17|  Chennai|       220322|          3|          7|
|         34|Cust34|  Chennai|       213216|          2|          5|
|         41|Cust41|   Mumbai|       154246|          2|          4|
|         23|Cust23|Hyderabad|       147509|          3|          6|
|         50|Cust50|Hyderabad|       144497|          2|          5|
|         37|Cust37|Bangalore|       135511|          2|          4|
|          4| Cust4|   Mumbai|       133266|          2|          4|
|          5| Cust5|Bangalore|       126977|          2|          3|
+-----------+------+---------+-------------+-------

### PHASE 5 – DEALER ANALYTICS

In [0]:
print("=" * 60)
print("PHASE 5 – DEALER ANALYTICS")
print("=" * 60)

PHASE 5 – DEALER ANALYTICS
+---------+-----------+-----------+-------------+-----------+-----------+
|dealer_id|dealer_name|dealer_city|total_revenue|total_sales|total_units|
+---------+-----------+-----------+-------------+-----------+-----------+
|        2|    Dealer2|    Chennai|       721442|          8|         21|
|        9|    Dealer9|    Chennai|       375896|          6|         14|
|        6|    Dealer6|  Bangalore|       342321|          4|         11|
|        1|    Dealer1|  Hyderabad|       320321|          8|         11|
|        5|    Dealer5|  Bangalore|       290639|          7|         14|
|        7|    Dealer7|  Hyderabad|       267090|          6|         12|
|        4|    Dealer4|     Mumbai|       220420|          8|         15|
|       10|   Dealer10|    Chennai|       218183|          6|         12|
|        8|    Dealer8|     Mumbai|        87418|          5|          8|
|        3|    Dealer3|     Mumbai|        81563|          5|          8|
+---------+

#### 5.1 Revenue per Dealer

In [0]:
dealer_revenue = df.groupBy("dealer_id", "dealer_name", "dealer_city") \
    .agg(sum("revenue").alias("total_revenue"),
         count("sale_id").alias("total_sales"),
         sum("quantity").alias("total_units")) \
    .orderBy(col("total_revenue").desc())

dealer_revenue.show()

PHASE 5 – DEALER ANALYTICS
+---------+-----------+-----------+-------------+-----------+-----------+
|dealer_id|dealer_name|dealer_city|total_revenue|total_sales|total_units|
+---------+-----------+-----------+-------------+-----------+-----------+
|        2|    Dealer2|    Chennai|       721442|          8|         21|
|        9|    Dealer9|    Chennai|       375896|          6|         14|
|        6|    Dealer6|  Bangalore|       342321|          4|         11|
|        1|    Dealer1|  Hyderabad|       320321|          8|         11|
|        5|    Dealer5|  Bangalore|       290639|          7|         14|
|        7|    Dealer7|  Hyderabad|       267090|          6|         12|
|        4|    Dealer4|     Mumbai|       220420|          8|         15|
|       10|   Dealer10|    Chennai|       218183|          6|         12|
|        8|    Dealer8|     Mumbai|        87418|          5|          8|
|        3|    Dealer3|     Mumbai|        81563|          5|          8|
+---------+

#### 5.2 Top 3 Dealers

In [0]:
print("\n--- Top 3 Dealers ---")
dealer_revenue.limit(3).show()

PHASE 5 – DEALER ANALYTICS
+---------+-----------+-----------+-------------+-----------+-----------+
|dealer_id|dealer_name|dealer_city|total_revenue|total_sales|total_units|
+---------+-----------+-----------+-------------+-----------+-----------+
|        2|    Dealer2|    Chennai|       721442|          8|         21|
|        9|    Dealer9|    Chennai|       375896|          6|         14|
|        6|    Dealer6|  Bangalore|       342321|          4|         11|
|        1|    Dealer1|  Hyderabad|       320321|          8|         11|
|        5|    Dealer5|  Bangalore|       290639|          7|         14|
|        7|    Dealer7|  Hyderabad|       267090|          6|         12|
|        4|    Dealer4|     Mumbai|       220420|          8|         15|
|       10|   Dealer10|    Chennai|       218183|          6|         12|
|        8|    Dealer8|     Mumbai|        87418|          5|          8|
|        3|    Dealer3|     Mumbai|        81563|          5|          8|
+---------+

#### 5.3 Dealer City Contribution

In [0]:
print("\n--- Dealer City Contribution ---")
city_total = dealer_revenue.groupBy("dealer_city") \
    .agg(sum("total_revenue").alias("city_total"))

dealer_city_contribution = dealer_revenue.join(city_total, "dealer_city") \
    .withColumn("pct_contribution",
                round(col("total_revenue") / col("city_total") * 100, 2))

dealer_city_contribution.show()

PHASE 5 – DEALER ANALYTICS
+---------+-----------+-----------+-------------+-----------+-----------+
|dealer_id|dealer_name|dealer_city|total_revenue|total_sales|total_units|
+---------+-----------+-----------+-------------+-----------+-----------+
|        2|    Dealer2|    Chennai|       721442|          8|         21|
|        9|    Dealer9|    Chennai|       375896|          6|         14|
|        6|    Dealer6|  Bangalore|       342321|          4|         11|
|        1|    Dealer1|  Hyderabad|       320321|          8|         11|
|        5|    Dealer5|  Bangalore|       290639|          7|         14|
|        7|    Dealer7|  Hyderabad|       267090|          6|         12|
|        4|    Dealer4|     Mumbai|       220420|          8|         15|
|       10|   Dealer10|    Chennai|       218183|          6|         12|
|        8|    Dealer8|     Mumbai|        87418|          5|          8|
|        3|    Dealer3|     Mumbai|        81563|          5|          8|
+---------+

### PHASE 6 – SQL ANALYSIS

In [0]:
print("=" * 60)
print("PHASE 6 – SQL ANALYSIS")
print("=" * 60)

PHASE 6 – SQL ANALYSIS
+-----------+------+---------+-------------+-----------+-----------+---+
|customer_id|  name|     city|total_revenue|total_sales|total_units|rnk|
+-----------+------+---------+-------------+-----------+-----------+---+
|         37|Cust37|Bangalore|       135511|          2|          4|  1|
|          5| Cust5|Bangalore|       126977|          2|          3|  2|
|         12|Cust12|Bangalore|        65964|          1|          3|  3|
|         13|Cust13|  Chennai|       230502|          5|         11|  1|
|         17|Cust17|  Chennai|       220322|          3|          7|  2|
|         34|Cust34|  Chennai|       213216|          2|          5|  3|
|         24|Cust24|Hyderabad|       233449|          3|          7|  1|
|         23|Cust23|Hyderabad|       147509|          3|          6|  2|
|         50|Cust50|Hyderabad|       144497|          2|          5|  3|
|         41|Cust41|   Mumbai|       154246|          2|          4|  1|
|          4| Cust4|   Mumba

#### Register Views

In [0]:
df.createOrReplaceTempView("sales_full")
customer_revenue.createOrReplaceTempView("customer_revenue")

PHASE 6 – SQL ANALYSIS
+-----------+------+---------+-------------+-----------+-----------+---+
|customer_id|  name|     city|total_revenue|total_sales|total_units|rnk|
+-----------+------+---------+-------------+-----------+-----------+---+
|         37|Cust37|Bangalore|       135511|          2|          4|  1|
|          5| Cust5|Bangalore|       126977|          2|          3|  2|
|         12|Cust12|Bangalore|        65964|          1|          3|  3|
|         13|Cust13|  Chennai|       230502|          5|         11|  1|
|         17|Cust17|  Chennai|       220322|          3|          7|  2|
|         34|Cust34|  Chennai|       213216|          2|          5|  3|
|         24|Cust24|Hyderabad|       233449|          3|          7|  1|
|         23|Cust23|Hyderabad|       147509|          3|          6|  2|
|         50|Cust50|Hyderabad|       144497|          2|          5|  3|
|         41|Cust41|   Mumbai|       154246|          2|          4|  1|
|          4| Cust4|   Mumba

#### Top 3 Customers per City

In [0]:
top3 = spark.sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY city ORDER BY total_revenue DESC) rnk
    FROM customer_revenue
)
WHERE rnk <= 3
""")

top3.show()

PHASE 6 – SQL ANALYSIS
+-----------+------+---------+-------------+-----------+-----------+---+
|customer_id|  name|     city|total_revenue|total_sales|total_units|rnk|
+-----------+------+---------+-------------+-----------+-----------+---+
|         37|Cust37|Bangalore|       135511|          2|          4|  1|
|          5| Cust5|Bangalore|       126977|          2|          3|  2|
|         12|Cust12|Bangalore|        65964|          1|          3|  3|
|         13|Cust13|  Chennai|       230502|          5|         11|  1|
|         17|Cust17|  Chennai|       220322|          3|          7|  2|
|         34|Cust34|  Chennai|       213216|          2|          5|  3|
|         24|Cust24|Hyderabad|       233449|          3|          7|  1|
|         23|Cust23|Hyderabad|       147509|          3|          6|  2|
|         50|Cust50|Hyderabad|       144497|          2|          5|  3|
|         41|Cust41|   Mumbai|       154246|          2|          4|  1|
|          4| Cust4|   Mumba

#### Monthly Trends

In [0]:
monthly = spark.sql("""
SELECT DATE_FORMAT(sale_date,'yyyy-MM') month,
       SUM(revenue) revenue
FROM sales_full
GROUP BY month
""")

monthly.show()

PHASE 6 – SQL ANALYSIS
+-----------+------+---------+-------------+-----------+-----------+---+
|customer_id|  name|     city|total_revenue|total_sales|total_units|rnk|
+-----------+------+---------+-------------+-----------+-----------+---+
|         37|Cust37|Bangalore|       135511|          2|          4|  1|
|          5| Cust5|Bangalore|       126977|          2|          3|  2|
|         12|Cust12|Bangalore|        65964|          1|          3|  3|
|         13|Cust13|  Chennai|       230502|          5|         11|  1|
|         17|Cust17|  Chennai|       220322|          3|          7|  2|
|         34|Cust34|  Chennai|       213216|          2|          5|  3|
|         24|Cust24|Hyderabad|       233449|          3|          7|  1|
|         23|Cust23|Hyderabad|       147509|          3|          6|  2|
|         50|Cust50|Hyderabad|       144497|          2|          5|  3|
|         41|Cust41|   Mumbai|       154246|          2|          4|  1|
|          4| Cust4|   Mumba

#### Repeat Customers

In [0]:
repeat = spark.sql("""
SELECT customer_id, COUNT(*) cnt
FROM sales_full
GROUP BY customer_id
HAVING cnt > 1
""")

repeat.show()

PHASE 6 – SQL ANALYSIS
+-----------+------+---------+-------------+-----------+-----------+---+
|customer_id|  name|     city|total_revenue|total_sales|total_units|rnk|
+-----------+------+---------+-------------+-----------+-----------+---+
|         37|Cust37|Bangalore|       135511|          2|          4|  1|
|          5| Cust5|Bangalore|       126977|          2|          3|  2|
|         12|Cust12|Bangalore|        65964|          1|          3|  3|
|         13|Cust13|  Chennai|       230502|          5|         11|  1|
|         17|Cust17|  Chennai|       220322|          3|          7|  2|
|         34|Cust34|  Chennai|       213216|          2|          5|  3|
|         24|Cust24|Hyderabad|       233449|          3|          7|  1|
|         23|Cust23|Hyderabad|       147509|          3|          6|  2|
|         50|Cust50|Hyderabad|       144497|          2|          5|  3|
|         41|Cust41|   Mumbai|       154246|          2|          4|  1|
|          4| Cust4|   Mumba

### PHASE 7 – OUTPUT

In [0]:
print("=" * 60)
print("PHASE 7 – OUTPUT")
print("=" * 60)

output_path = "/Volumes/workspace/default/day6_files/output/"

PHASE 7 – OUTPUT
All outputs saved to /Volumes/workspace/default/day6_files/output/


#### Save Outputs

In [0]:
customer_revenue.write.mode("overwrite").option("header", True).csv(output_path + "customer_revenue")
brand_sales.write.mode("overwrite").option("header", True).csv(output_path + "brand_sales")
city_revenue.write.mode("overwrite").option("header", True).csv(output_path + "city_revenue")
dealer_revenue.write.mode("overwrite").option("header", True).csv(output_path + "dealer_revenue")
top3.write.mode("overwrite").option("header", True).csv(output_path + "top3")
monthly.write.mode("overwrite").option("header", True).csv(output_path + "monthly")
repeat.write.mode("overwrite").option("header", True).csv(output_path + "repeat")

print(f"All outputs saved to {output_path}")

PHASE 7 – OUTPUT
All outputs saved to /Volumes/workspace/default/day6_files/output/
